# Урок 09 - Патерн проектування метакогніції


## Setup

This notebook demonstrates the Metacognition design pattern using the Microsoft Agent Framework.

**Prerequisites:**
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Що таке метакогніція?

Метакогніція — це **мислення про мислення**. У контексті агентів штучного інтелекту це означає створення агентів, які можуть:

- **Саморефлексувати** над власними результатами та процесом міркування
- **Виявляти помилки** та плавно відновлюватися замість того, щоб мовчки зазнавати невдачі
- **Оцінювати**, чи є їхні відповіді повними та корисними
- **Адаптуватися** у своїй стратегії, якщо початковий підхід не працює (наприклад, переходити на резервну систему)

Метакогнітивний агент не просто відповідає на запитання — він контролює власну ефективність і коригує свою поведінку у реальному часі.


## Основні та запасні інструменти

Загальновживаним метапізнавальним шаблоном є **стратегія запасного варіанта**. Агент спочатку намагається використати основний інструмент; якщо він не вдається (наприклад, помилка 404), агент усвідомлює невдачу і прозоро переключається на запасний інструмент.

Це відображає системи реального світу, де основні сервіси можуть бути недоступні, і агенти повинні самостійно діагностувати проблему перед вибором альтернативного шляху.

Нижче ми визначаємо два інструменти пошуку рейсів:
- **Основний** — покриває Париж, Токіо та Барселону
- **Запасний** — покриває Берлін, Сідней та Нью-Йорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Агент із саморефлексією та відновленням після помилок

Нижченаведений агент отримує інструкції спочатку спробувати основну систему управління польотом, розпізнавати збої та прозоро переходити на резервну систему. Після кожної відповіді він коротко оцінює себе, чи повністю відповів на запитання користувача.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Шаблон самооцінки

Іншим аспектом метакогніції є **самооцінка**: окремий агент (або той самий агент при другому проході) переглядає відповідь на повноту, точність і корисність.

Нижче ми створюємо агента `ResponseEvaluator`, який оцінює відповіді туристичного агента за трьома вимірами.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резюме

У цьому уроці ви дізналися, як створювати **метакогнітивні агенти** за допомогою Microsoft Agent Framework:

- **Саморефлексія**: Агенти, які контролюють власне мислення і прозоро повідомляють про те, що відбулося.
- **Відновлення після помилки з запасними варіантами**: Шаблон з основним і запасним інструментом, коли агент виявляє помилки (наприклад, 404) і автоматично пробує альтернативне джерело.
- **Самооцінка**: Окремий агент-оцінювач, який оцінює відповіді за повнотою, точністю та корисністю.

Ці шаблони роблять агентів більш надійними, прозорими та довіреними — критично важливими якостями для виробничого розгортання.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
